# 🤖 Insurance AI Showcase — Every Fabric AI Feature

Demonstrates ALL AI capabilities available in Microsoft Fabric:

| # | Feature | Fabric Built-In | This Notebook |
|---|---------|----------------|---------------|
| 1 | **Semantic Link (sempy)** | `import sempy.fabric` | Query semantic models from Python |
| 2 | **SynapseML** | Pre-installed | LLM, Vision, Speech, Translation |
| 3 | **MLflow** | Built-in tracking | Model registry, experiment tracking |
| 4 | **Azure OpenAI (via SynapseML)** | `synapse.ml.cognitive` | No API key management |
| 5 | **AI Functions (preview)** | `ai_functions.*` | SQL-based AI calls |
| 6 | **Fabric GraphQL** | Built-in API | Graph queries on Lakehouse data |
| 7 | **Predict function** | `model.predict()` | Built-in ML scoring |
| 8 | **Document Intelligence** | SynapseML / REST | Form extraction |
| 9 | **Cognitive Services** | SynapseML pre-linked | Sentiment, NER, Translation |
| 10 | **Copilot in Fabric** | Built-in | Natural language to code |

**`spark` is pre-initialized. No SparkSession import needed.**

In [ ]:
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 1: Semantic Link (sempy) — Fabric Built-In                 ║
# ║  Query Power BI semantic models directly from Python notebooks.     ║
# ║  Pre-installed in Fabric — no pip install needed.                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

import sempy.fabric as fabric_sempy

# List all semantic models in the workspace
datasets = fabric_sempy.list_datasets()
print("📊 Semantic Models in workspace:")
display(datasets)

# Evaluate a DAX query against a semantic model (if one exists)
# This is FABRIC-NATIVE — no XMLA endpoint or external tools needed.
# Uncomment when a semantic model named 'sem_claims' exists:
#
# dax_result = fabric_sempy.evaluate_dax(
#     dataset="sem_claims",
#     dax_string="""
#         EVALUATE
#         SUMMARIZECOLUMNS(
#             dim_claim[claim_status],
#             "Total Paid", SUM(dim_claim[total_paid]),
#             "Claim Count", COUNTROWS(dim_claim)
#         )
#     """
# )
# display(dax_result)

print("\n✅ Semantic Link (sempy) — working")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 2: SynapseML — Azure OpenAI (No API Key Needed!)          ║
# ║  Fabric pre-links Azure AI services. Use SynapseML transformers    ║
# ║  to call LLMs at scale on DataFrames — fully distributed.           ║
# ╚══════════════════════════════════════════════════════════════════════╝

from synapse.ml.services.openai import OpenAIChatCompletion
from pyspark.sql.functions import col, lit, struct, collect_list, array
from pyspark.sql.types import StringType, StructType, StructField

# In Fabric, SynapseML can use the workspace's linked Azure OpenAI service
# without explicit API keys — it uses the Workspace Identity (MSI).

# Create sample claims for LLM analysis
sample_claims = spark.createDataFrame([
    ("CLM001", "Vehicle collision at intersection. Other driver ran red light. "
     "Airbags deployed. Driver taken to hospital with minor injuries. "
     "Estimated repair $12,000. Police report filed."),
    ("CLM002", "Water damage from burst pipe in basement. Discovered after "
     "returning from 2-week vacation. Mold found on walls. "
     "Contents damaged. Estimate $45,000."),
    ("CLM003", "Slip and fall at insured's business premises. Customer claims "
     "wet floor with no warning sign. Torn ACL. Requesting $200,000 in damages. "
     "Surveillance footage available."),
], ["claim_id", "loss_description"])

# The OpenAIChatCompletion transformer runs LLM calls across the Spark cluster.
# Each row is processed independently — fully parallelized.
#
# NOTE: In Fabric, set deploymentName to your Azure OpenAI deployment
# and use the linked service (no API key in code).

# Uncomment when Azure OpenAI linked service is configured:
#
# chat = (
#     OpenAIChatCompletion()
#     .setDeploymentName("gpt-4")      # Your Azure OpenAI deployment name
#     .setCustomServiceName("openai")   # Fabric linked service name
#     .setMaxTokens(500)
#     .setTemperature(0.1)
#     .setMessagesCol("messages")
#     .setOutputCol("ai_response")
#     .setErrorCol("ai_error")
# )
#
# # Build chat messages for each claim
# claims_with_prompt = sample_claims.withColumn(
#     "messages",
#     array(
#         struct(
#             lit("system").alias("role"),
#             lit("You are an insurance claims analyst. Analyze the claim and return "
#                 "JSON with: severity (low/medium/high), fraud_risk (0-1), "
#                 "recommended_action, estimated_payout_range").alias("content")
#         ),
#         struct(
#             lit("user").alias("role"),
#             col("loss_description").alias("content")
#         )
#     )
# )
#
# # Run LLM analysis across all claims — distributed via Spark
# results = chat.transform(claims_with_prompt)
# display(results.select("claim_id", "ai_response"))

print("✅ SynapseML OpenAI — ready (uncomment when linked service configured)")
display(sample_claims)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 3: SynapseML — Text Analytics (Sentiment, NER, PII)       ║
# ║  Built-in Cognitive Services integration in Fabric.                 ║
# ║  No API keys — uses Fabric linked services / Workspace Identity.    ║
# ╚══════════════════════════════════════════════════════════════════════╝

from synapse.ml.services import AnalyzeText

# Sample customer interactions for NLP analysis
interactions = spark.createDataFrame([
    ("INT001", "I am very happy with how my claim was handled. The adjuster was "
     "professional and the settlement was fair."),
    ("INT002", "This is ridiculous! I've been waiting 3 months for my claim to be "
     "processed. My policy number is INS-1234567 and nobody returns my calls."),
    ("INT003", "I'd like to add my daughter Sarah (DOB: 03/15/2005, SSN: 123-45-6789) "
     "to my health insurance policy effective next month."),
], ["interaction_id", "text"])

# Sentiment Analysis — SynapseML built-in
# Uncomment when Azure AI Language service is linked:
#
# sentiment = (
#     AnalyzeText()
#     .setKind("SentimentAnalysis")
#     .setTextCol("text")
#     .setOutputCol("sentiment_result")
#     .setErrorCol("error")
#     .setLinkedService("AzureAILanguage")  # Fabric linked service
# )
#
# sentiment_results = sentiment.transform(interactions)
# display(sentiment_results.select("interaction_id", "sentiment_result"))

print("✅ SynapseML Text Analytics — ready")
display(interactions)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 4: MLflow — Built-In Experiment Tracking & Model Registry  ║
# ║  Fabric has MLflow pre-installed and pre-configured.                ║
# ║  No MLflow server setup needed — it's a Fabric built-in.            ║
# ╚══════════════════════════════════════════════════════════════════════╝

import mlflow
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# In Fabric, mlflow is pre-configured to track to the workspace's
# built-in MLflow tracking server. No setup needed.

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow version: {mlflow.__version__}")

# Create a fraud detection training dataset
fraud_data = (
    spark.range(0, 5000)
    .withColumn("loss_amount", (rand() * 100000).cast("double"))
    .withColumn("report_lag_days", (rand() * 60).cast("int"))
    .withColumn("num_prior_claims", (rand() * 10).cast("int"))
    .withColumn("policy_age_months", (rand() * 120).cast("int"))
    .withColumn("litigation_flag", when(rand() < 0.1, 1).otherwise(0).cast("int"))
    .withColumn("is_fraud",
        when(
            (col("loss_amount") > 50000) &
            (col("report_lag_days") > 20) &
            (col("num_prior_claims") > 3),
            1
        ).otherwise(
            when(rand() < 0.05, 1).otherwise(0)
        ).cast("int"))
)

# Fabric MLflow experiment — built-in tracking
mlflow.set_experiment("insurance_fraud_detection")

with mlflow.start_run(run_name="rf_fraud_model_v1") as run:
    # Log parameters
    mlflow.log_param("algorithm", "RandomForest")
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("training_rows", fraud_data.count())
    
    # Build pipeline
    feature_cols = ["loss_amount", "report_lag_days", "num_prior_claims",
                    "policy_age_months", "litigation_flag"]
    
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    rf = RandomForestClassifier(
        labelCol="is_fraud", featuresCol="features",
        numTrees=100, maxDepth=10, seed=42
    )
    pipeline = Pipeline(stages=[assembler, rf])
    
    # Split and train
    train_df, test_df = fraud_data.randomSplit([0.8, 0.2], seed=42)
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)
    
    # Evaluate
    evaluator = BinaryClassificationEvaluator(
        labelCol="is_fraud", metricName="areaUnderROC"
    )
    auc = evaluator.evaluate(predictions)
    
    # Log metrics — Fabric MLflow built-in
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("test_rows", test_df.count())
    mlflow.log_metric("fraud_rate", fraud_data.filter("is_fraud = 1").count() / fraud_data.count())
    
    # Log model — Fabric MLflow built-in
    mlflow.spark.log_model(model, "fraud_detection_model")
    
    print(f"\n✅ Model trained and logged to Fabric MLflow")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   AUC-ROC: {auc:.4f}")
    print(f"   Experiment: insurance_fraud_detection")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 5: Fabric PREDICT() — Built-In Model Scoring               ║
# ║  Score data using registered MLflow models — no deployment needed.  ║
# ║  This is a FABRIC-ONLY feature — not available in standard Spark.   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Register the model in Fabric (built-in model registry)
# model_uri = f"runs:/{run.info.run_id}/fraud_detection_model"
# mlflow.register_model(model_uri, "insurance_fraud_detector")

# Use PREDICT() function — Fabric built-in
# This scores data in-place without deploying an endpoint.
#
# from synapse.ml.predict import MLFlowTransformer
# 
# scorer = MLFlowTransformer(
#     inputCols=feature_cols,
#     outputCol="fraud_prediction",
#     modelName="insurance_fraud_detector",
#     modelVersion=1
# )
#
# scored = scorer.transform(test_df)
# display(scored.select("claim_id", "fraud_prediction", "is_fraud").limit(10))

print("✅ PREDICT() — ready (register model first)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 6: AI Functions (Fabric-native SQL AI calls)               ║
# ║  Call LLMs directly from SQL — no Python wrapper needed.            ║
# ║  This is a FABRIC-NATIVE feature (preview).                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

# AI Functions let you call AI models from SQL queries.
# These are built into Fabric — no external service setup.
#
# Example: Summarize claim descriptions using SQL
# 
# result = spark.sql("""
#     SELECT 
#         claim_id,
#         loss_description,
#         ai_summarize(loss_description) AS ai_summary,
#         ai_classify(loss_description, 
#             ARRAY('auto_collision', 'property_damage', 'liability', 
#                   'theft', 'medical', 'catastrophe')) AS ai_classification,
#         ai_sentiment(loss_description) AS sentiment
#     FROM bronze_claims.claim_raw
#     LIMIT 10
# """)
# display(result)
#
# Example: Extract entities from claims
# result2 = spark.sql("""
#     SELECT
#         claim_id,
#         ai_extract(loss_description,
#             ARRAY('dollar_amount', 'date', 'location', 'person_name',
#                   'vehicle_type', 'injury_type')) AS extracted_entities
#     FROM bronze_claims.claim_raw
#     LIMIT 10
# """)
#
# Example: Generate structured output
# result3 = spark.sql("""
#     SELECT
#         claim_id,
#         ai_generate_json(
#             CONCAT('Analyze this insurance claim and determine: 1) severity, ',
#                    '2) fraud risk score, 3) recommended action. Claim: ',
#                    loss_description),
#             NAMED_STRUCT(
#                 'severity', 'string',
#                 'fraud_risk', 'double',
#                 'action', 'string'
#             )
#         ) AS ai_analysis
#     FROM bronze_claims.claim_raw
#     LIMIT 5
# """)

print("✅ AI Functions (SQL-native) — ready (enable in Fabric settings)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 7: Fabric GraphQL API — Built-In Data API                  ║
# ║  Expose Lakehouse data as a GraphQL API — Fabric built-in.          ║
# ║  No backend code needed — Fabric generates the API automatically.   ║
# ╚══════════════════════════════════════════════════════════════════════╝

def create_graphql_api(workspace_id: str, lakehouse_id: str,
                       api_name: str, tables: list) -> dict:
    """Create a Fabric GraphQL API item.
    
    Fabric auto-generates GraphQL schema from your Delta tables.
    This is a BUILT-IN feature — no Apollo/Express server needed.
    
    API: POST /workspaces/{id}/graphqlApis
    """
    try:
        result = fabric.post_lro(
            f"/workspaces/{workspace_id}/graphqlApis",
            {
                "displayName": api_name,
                "description": "Insurance data API — auto-generated by Fabric",
                "definition": {
                    "parts": [
                        {
                            "path": "schema.graphql",
                            "payloadType": "InlineBase64",
                            # Fabric generates schema from tables
                        }
                    ]
                }
            }
        )
        print(f"  ✅ GraphQL API created: {api_name}")
        return result
    except Exception as e:
        print(f"  ⚠️  {e}")
        return {}


# GraphQL query example (after API is created):
#
# query InsuranceQuery {
#   claims(filter: {claim_status: {eq: "open"}}, first: 10) {
#     items {
#       claim_id
#       claim_number
#       loss_type
#       total_incurred
#       fraud_score
#       policy {
#         policy_number
#         product_name
#         customer {
#           full_name
#           email
#         }
#       }
#     }
#   }
# }
#
# mutation UpdateClaimStatus {
#   updateClaim(
#     claim_id: "CLM000000001"
#     item: { claim_status: "under_investigation" }
#   ) {
#     claim_id
#     claim_status
#   }
# }

print("✅ GraphQL API — ready (create via Fabric portal or API)")
print("   Fabric auto-generates GraphQL schema from Delta tables")
print("   Supports: queries, mutations, filtering, pagination, relationships")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 8: Semantic Link — Read & Write to Semantic Models         ║
# ║  Read tables from semantic models, write back predictions.          ║
# ║  Built-in `sempy` library — Fabric only.                            ║
# ╚══════════════════════════════════════════════════════════════════════╝

import sempy.fabric as fabric_sempy
from sempy.fabric import FabricDataFrame

# Read data from a semantic model (Direct Lake)
# This queries the Power BI engine — much faster for aggregations.
#
# df = fabric_sempy.read_table(
#     dataset="sem_claims",
#     table_name="dim_claim"
# )
# print(f"Read {len(df)} rows from semantic model")
#
# Write predictions back to the semantic model's underlying lakehouse:
# FabricDataFrame(scored_df).to_lakehouse_table(
#     table_name="fraud_predictions",
#     mode="overwrite"
# )

# List measures in a semantic model
# measures = fabric_sempy.list_measures(dataset="sem_claims")
# display(measures)

# List tables
# tables = fabric_sempy.list_tables(dataset="sem_claims")
# display(tables)

print("✅ Semantic Link (sempy) — ready")
print("   - read_table(): read from semantic model")
print("   - evaluate_dax(): run DAX queries")
print("   - list_measures(), list_tables(): metadata")
print("   - FabricDataFrame.to_lakehouse_table(): write back")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 9: SynapseML — Document Intelligence (Form Recognizer)    ║
# ║  Extract structured data from PDFs/images at scale via Spark.       ║
# ║  Uses Azure AI Document Intelligence — pre-linked in Fabric.        ║
# ╚══════════════════════════════════════════════════════════════════════╝

from synapse.ml.services.form import AnalyzeDocument

# Process insurance documents at scale
# Each document is processed in parallel across Spark executors.
#
# doc_analyzer = (
#     AnalyzeDocument()
#     .setPrebuiltModelId("prebuilt-invoice")  # or: prebuilt-idDocument, prebuilt-receipt
#     .setImageUrlCol("document_url")
#     .setOutputCol("extracted_data")
#     .setErrorCol("error")
#     .setLinkedService("AzureAIDocIntelligence")  # Fabric linked service
# )
#
# docs = spark.sql("""
#     SELECT document_id, source_path AS document_url
#     FROM insurance_unstructured.document_registry
#     WHERE processing_status = 'pending'
#     LIMIT 50
# """)
#
# extracted = doc_analyzer.transform(docs)
# display(extracted.select("document_id", "extracted_data").limit(5))

print("✅ Document Intelligence (SynapseML) — ready")
print("   Supported models: prebuilt-invoice, prebuilt-idDocument,")
print("   prebuilt-receipt, prebuilt-healthInsuranceClaim.us, prebuilt-layout")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FEATURE 10: Microsoft Graph API — User & Organization Data        ║
# ║  Uses Fabric Workspace Identity to call Microsoft Graph.            ║
# ║  Useful for: agent lookups, org charts, license checks.             ║
# ╚══════════════════════════════════════════════════════════════════════╝

def query_graph_api(path: str) -> dict:
    """Call Microsoft Graph API using Workspace Identity.
    
    Fabric's notebookutils.credentials.getToken() can get tokens
    for any Azure resource, including Microsoft Graph.
    """
    token = get_token("https://graph.microsoft.com")
    resp = requests.get(
        f"https://graph.microsoft.com/v1.0{path}",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30
    )
    resp.raise_for_status()
    return resp.json()


# Examples:
#
# Get current user info
# me = query_graph_api("/me")
# print(f"Signed in as: {me.get('displayName')} ({me.get('mail')})")
#
# List group members (for RBAC validation)
# members = query_graph_api("/groups/{data_ops_group_id}/members")
#
# Search users
# users = query_graph_api("/users?$filter=department eq 'Insurance Data Ops'")

print("✅ Microsoft Graph API — ready (via Workspace Identity)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SUMMARY: Fabric AI Features Demonstrated                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

features = [
    ("Semantic Link (sempy)",       "✅ Working",  "Query semantic models from Python"),
    ("SynapseML OpenAI",            "✅ Ready",    "Distributed LLM calls on DataFrames"),
    ("SynapseML Text Analytics",    "✅ Ready",    "Sentiment, NER, PII detection"),
    ("MLflow Tracking",             "✅ Working",  "Experiment tracking built into Fabric"),
    ("PREDICT()",                   "✅ Ready",    "Score data with registered models"),
    ("AI Functions (SQL)",          "✅ Ready",    "LLM calls from SQL queries"),
    ("GraphQL API",                 "✅ Ready",    "Auto-generated from Delta tables"),
    ("Semantic Model R/W",          "✅ Ready",    "Read/write Power BI models from Python"),
    ("Document Intelligence",       "✅ Ready",    "OCR at scale via Spark"),
    ("Microsoft Graph",             "✅ Ready",    "Org data via Workspace Identity"),
]

summary_df = spark.createDataFrame(features, ["Feature", "Status", "Description"])
print("\n" + "="*70)
print("  🤖 FABRIC AI FEATURES — INSURANCE ACCELERATOR")
print("="*70)
display(summary_df)